# 🇻🇳 Huấn luyện YOLO26 Nhận Diện & Tự Động Tính Tổng Tiền Việt Nam

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bentovnace/Yolo26-Vietnamese-Money/blob/main/notebooks/train_yolo26_money_vn.ipynb)

Sổ tay hướng dẫn huấn luyện mô hình **YOLO26** (kiến trúc NMS-Free End-to-End từ Ultralytics) trên tập dữ liệu Roboflow **Money_VN_Regconization 2**.

### 📌 Thông tin tập dữ liệu:
- **Nguồn:** Roboflow `blindtrix/money_vn_regconization-2-cumhs` (Version 2)
- **Tổng số ảnh:** 10,116 ảnh (Train: 9,129 | Valid: 563 | Test: 424)
- **9 Mệnh giá tiền:** 1.000đ, 2.000đ, 5.000đ, 10.000đ, 20.000đ, 50.000đ, 100.000đ, 200.000đ, 500.000đ
- **Định dạng tải:** `yolo26`


## 1. Kiểm tra cấu hình GPU trên Google Colab
Đảm bảo Colab đang sử dụng phần cứng GPU (T4, V100, L4, A100).

In [ ]:
!nvidia-smi

## 2. Cài đặt các thư viện cần thiết
Cài đặt phiên bản mới nhất của `ultralytics` (hỗ trợ YOLO26), `roboflow`, `wandb` (để theo dõi số liệu trực tuyến), `opencv-python`, `supervision`.

In [ ]:
!pip install --upgrade roboflow ultralytics opencv-python matplotlib supervision pyyaml wandb

## 3. Tải tập dữ liệu từ Roboflow (Kèm cơ chế Overwrite & Tự động khôi phục)
Sử dụng `overwrite=True` để tránh lỗi tải dở hoặc thư mục rỗng do Roboflow cache.

In [ ]:
import os
import zipfile
from pathlib import Path
from roboflow import Roboflow

# Khởi tạo Roboflow client
rf = Roboflow(api_key="FtXoE5Gc1n3NlmiAnXvn")
project = rf.workspace("blindtrix").project("money_vn_regconization-2-cumhs")
version = project.version(2)

# Tải dataset với format yolo26 và overwrite=True để đảm bảo tải & giải nén trọn vẹn
dataset = version.download("yolo26", overwrite=True)
print("\n[+] Dataset location:", dataset.location)

# Kiểm tra nếu còn file roboflow.zip chưa giải nén xong
loc_path = Path(dataset.location)
zip_in_loc = loc_path / "roboflow.zip"
if zip_in_loc.exists():
    print("[*] Đang giải nén roboflow.zip hoàn tất...")
    with zipfile.ZipFile(zip_in_loc, 'r') as zf:
        zf.extractall(loc_path)
    os.remove(zip_in_loc)
    print("[+] Giải nén thành công!")


## 4. Kiểm tra cấu hình `data.yaml` & hiển thị ảnh mẫu
Ô này tự động tìm kiếm file `data.yaml` và tự động tạo lại đường dẫn tuyệt đối chuẩn xác cho YOLO26.

In [ ]:
import yaml
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

# Tìm kiếm data.yaml trong thư mục dataset hoặc toàn bộ /content
loc_path = Path(dataset.location)
yaml_candidates = list(loc_path.glob("*.yaml")) + list(loc_path.rglob("data.yaml")) + list(Path("/content").rglob("*data.yaml"))

if not yaml_candidates:
    print("[!] data.yaml chưa có sẵn, đang tự động tạo file data.yaml chuẩn...")
    data_yaml_path = loc_path / "data.yaml"
    default_config = {
        "train": str(loc_path / "train" / "images"),
        "val": str(loc_path / "valid" / "images"),
        "test": str(loc_path / "test" / "images"),
        "nc": 9,
        "names": ["1000", "10000", "100000", "2000", "20000", "200000", "5000", "50000", "500000"]
    }
    with open(data_yaml_path, "w") as f:
        yaml.dump(default_config, f, sort_keys=False)
else:
    data_yaml_path = yaml_candidates[0]

print("[*] Đường dẫn data.yaml:", data_yaml_path)
with open(data_yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

# Cập nhật đường dẫn tuyệt đối để Ultralytics đọc không bị lệch
data_config["train"] = str(loc_path / "train" / "images")
data_config["val"] = str(loc_path / "valid" / "images")
data_config["test"] = str(loc_path / "test" / "images")
with open(data_yaml_path, "w") as f:
    yaml.dump(data_config, f, sort_keys=False)

print("[*] Danh sách các class mệnh giá:", data_config.get("names"))
print("[*] Số lượng classes:", len(data_config.get("names", [])))

train_imgs = list((loc_path / "train" / "images").glob("*.jpg"))[:3]
if train_imgs:
    fig, axes = plt.subplots(1, len(train_imgs), figsize=(15, 5))
    for ax, p in zip(axes, train_imgs):
        img = cv2.imread(str(p))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        ax.set_title(p.name, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 5. (Tùy chọn) Kết nối Weights & Biases (W&B) để theo dõi thời gian thực
Nếu bạn muốn xem biểu đồ loss, mAP, GPU metrics cập nhật trực tiếp theo thời gian thực trên web/điện thoại, hãy đăng nhập W&B (miễn phí):
- Lấy key tại: https://wandb.ai/authorize
- Nếu không dùng W&B, bạn có thể bỏ qua ô này hoặc chọn `disabled`.

In [ ]:
import wandb

# Đăng nhập W&B (Paste API Key từ https://wandb.ai/authorize vào ô nhập bên dưới)
# Nếu không muốn đồng bộ lên mạng, chọn option 3 (Don't visualize my results)
try:
    wandb.login()
    print("[+] W&B đã sẵn sàng! Ultralytics sẽ tự động đồng bộ số liệu qua từng epoch.")
except Exception as e:
    print("[*] Bỏ qua đăng nhập W&B, số liệu sẽ được lưu nội bộ trên máy:", e)


## 6. Khởi tạo & Huấn luyện mô hình YOLO26 (Tối ưu Max Speed trên GPU L4)
- `batch=64` hoặc `128`: Tận dụng trọn vẹn VRAM của L4.
- `cache='ram'`: Nạp toàn bộ 9.100 ảnh vào RAM để GPU chạy 100% không bị chờ ổ cứng.
- `patience=30`: Tránh dừng sớm khi mô hình đang học tốt.

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLO26 Nano (hoặc 'yolo26s.pt')
model = YOLO("yolo26n.pt")

# Bắt đầu huấn luyện
results = model.train(
    data=str(data_yaml_path),
    epochs=200,            # Số epochs tối đa
    imgsz=640,             # Kích thước ảnh chuẩn 640x640
    batch=64,              # hoặc 128 nếu muốn GPU ăn ~18-20GB VRAM
    cache='ram',           # Tăng tốc tối đa: giữ ảnh trên RAM
    patience=30,           # Early stopping
    workers=8,             # Đa luồng đọc ảnh
    optimizer="auto",      # Tự động chọn optimizer tối ưu (MuSGD/AdamW)
    save=True,
    project="yolo26_vietnamese_money",
    name="train_run_1",
    device=0,
    verbose=True
)

print("\n[+] Huấn luyện mô hình YOLO26 hoàn tất!")

## 7. Đánh giá mô hình & Hiển thị Biểu đồ Kết quả (Loss, mAP, Confusion Matrix)
Xem trực tiếp các biểu đồ huấn luyện và đo lường độ chính xác trên tập Test (Test Split).

In [ ]:
from IPython.display import Image, display

# 1. Đánh giá trên tập test
metrics = model.val(
    data=str(data_yaml_path),
    split="test",
    project="yolo26_vietnamese_money",
    name="test_eval"
)

print("\n========== KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ==========")
print(f"[*] mAP@50:     {metrics.box.map50:.4f}")
print(f"[*] mAP@50-95:  {metrics.box.map:.4f}")
print(f"[*] Precision:  {metrics.box.mp:.4f}")
print(f"[*] Recall:     {metrics.box.mr:.4f}")
print("====================================================\n")

# 2. Hiển thị biểu đồ kết quả train (results.png)
results_png = list(Path(".").glob("**/train_run_1/results.png")) + list(Path(".").glob("**/results.png"))
if results_png:
    print("[*] BIỂU ĐỒ LOSS & MAP QUA CÁC EPOCHS:")
    display(Image(filename=str(results_png[0]), width=950))

# 3. Hiển thị ma trận nhầm lẫn (Confusion Matrix)
cm_png = list(Path(".").glob("**/train_run_1/confusion_matrix.png")) + list(Path(".").glob("**/confusion_matrix.png"))
if cm_png:
    print("[*] MA TRẬN NHẦM LẪN (CONFUSION MATRIX):")
    display(Image(filename=str(cm_png[0]), width=750))


## 8. Thử nghiệm nhận diện & Tự động tính tổng tiền Việt Nam
Hàm dưới đây nhận diện các tờ tiền trong ảnh, nhóm theo từng mệnh giá và tính tổng số tiền (VNĐ).

In [ ]:
DENOMINATION_VALUES = {
    "1000": 1000,
    "2000": 2000,
    "5000": 5000,
    "10000": 10000,
    "20000": 20000,
    "50000": 50000,
    "100000": 100000,
    "200000": 200000,
    "500000": 500000
}

def detect_and_calculate_money(model_instance, image_path, conf=0.25):
    pred = model_instance.predict(image_path, conf=conf, verbose=False)[0]
    breakdown = {}
    total_vnd = 0
    
    for box in pred.boxes:
        cls_id = int(box.cls[0].item())
        cls_name = pred.names[cls_id]
        clean_cls = str(cls_name).replace('k', '000').replace('K', '000').strip()
        val = DENOMINATION_VALUES.get(clean_cls, 0)
        if val > 0:
            total_vnd += val
            breakdown[clean_cls] = breakdown.get(clean_cls, 0) + 1
            
    formatted_total = f"{total_vnd:,.0f} VNĐ".replace(",", ".")
    print(f"\n{'='*50}")
    print(f"[*] Ảnh: {Path(image_path).name}")
    print(f"[*] Tổng số tờ phát hiện: {len(pred.boxes)} tờ")
    print(f"[*] TỔNG TIỀN:            {formatted_total}")
    for denom, count in sorted(breakdown.items(), key=lambda x: int(x[0]), reverse=True):
        print(f"    - {int(denom):,}đ: {count} tờ = {count * int(denom):,}đ")
    print(f"{'='*50}")
    
    annotated_bgr = pred.plot()
    plt.figure(figsize=(9, 7))
    plt.imshow(cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB))
    plt.title(f"Tổng: {formatted_total} ({len(pred.boxes)} tờ)", fontsize=13, fontweight="bold")
    plt.axis("off")
    plt.show()

test_sample_images = list((Path(dataset.location) / "test" / "images").glob("*.jpg"))[:2]
for sample_p in test_sample_images:
    detect_and_calculate_money(model, str(sample_p), conf=0.3)

## 9. Xuất mô hình & Tải file trọng số `best.pt` về máy
Ô này sẽ tìm file `best.pt`, xuất mô hình sang ONNX và nén thành `yolo26_vietnamese_money_weights.zip` để tải tự động về máy tính của bạn.

Sau khi tải về:
1. Giải nén file zip.
2. Copy file `best.pt` vào thư mục `models/` trong project trên máy của bạn (`models/best.pt`).
3. Chạy `make build` và `make run` để khởi động Docker container!

In [ ]:
import shutil
from google.colab import files

# 1. Xuất thêm sang định dạng ONNX (tuỳ chọn cho suy luận CPU)
try:
    model.export(format="onnx")
    print("[+] Đã xuất mô hình sang ONNX!")
except Exception as e:
    print("[!] Bỏ qua ONNX export:", e)

# 2. Tìm file best.pt
best_candidates = list(Path(".").glob("**/train_run_1/weights/best.pt")) + list(Path(".").glob("**/best.pt"))
if not best_candidates:
    raise FileNotFoundError("Không tìm thấy file best.pt! Kiểm tra thư mục runs/")
best_weights = best_candidates[0]
print("[*] Trọng số tối ưu tại:", best_weights)

# 3. Tạo thư mục đóng gói triển khai (deploy)
deploy_dir = Path("deploy_package")
deploy_dir.mkdir(exist_ok=True)
shutil.copy(best_weights, deploy_dir / "best.pt")
if Path(data_yaml_path).exists():
    shutil.copy(data_yaml_path, deploy_dir / "data.yaml")

# 4. Đóng gói zip
zip_file = shutil.make_archive("yolo26_vietnamese_money_weights", "zip", deploy_dir)
print(f"[+] Đã tạo file: {zip_file}")

# 5. Tự động tải về máy
files.download(zip_file)
print("[+] Trình duyệt đang tải file về máy. Hãy copy best.pt vào models/ trên máy của bạn!")